# LensIQ: deploy a Roboflow Universe detector as a Databricks-native YOLO endpoint

Downloads the trained `weights.pt` for one Roboflow Universe model at
**deploy time** using the Roboflow Python SDK, bakes the weights into an
MLflow PyFunc as an artifact, registers it to Unity Catalog, and serves
it behind its own Databricks Model Serving endpoint. Inference runs
**locally inside the endpoint** via `ultralytics.YOLO` - there are no
HTTP calls to `serverless.roboflow.com` at serve time. Re-run this
notebook once per use case (license plate, spill, wet floor sign,
cigarette/vape, slip & fall) with different widgets - the bundle's
`lensiq_deploy_roboflow_detectors` job wires that up as a multi-task
fan-out.

Why one endpoint per use case (instead of one PyFunc dispatching by
`model_id`):

- Independent UC registry entry per model -> per-use-case versioning + RBAC.
- Independent serving config (workload size, scale-to-zero, alerts).
- Single-purpose endpoints make ownership / cost-attribution obvious.
- One endpoint can be updated without redeploying the others.

Why offline weights instead of proxying:

Every detector should run entirely on Databricks - no external API hop
at inference time. The Roboflow SDK's `model.download()` writes a
`weights.pt` file that we can hand to `ultralytics.YOLO()` and serve
just like the COCO YOLO in `deploy_yolo.ipynb`. Same payload, same
response shape, just no network call to a third-party vendor on every
frame.

Payload (matches AppKit `serving()` invoke):

```json
{"dataframe_records": [{"image": "<b64>", "conf": 0.35, "iou": 0.5}]}
```

Response (matches the YOLO endpoint so `_normalizeDatabricks` works):

```json
{"predictions": [[{"label": "...", "class_id": 0, "confidence": 0.8, "bbox": [x1,y1,x2,y2]}]]}
```

In [ ]:
dbutils.widgets.text("catalog", "reggie_pierce_7405614800873570")
dbutils.widgets.text("schema", "lensiq")
# Per-use-case slug. Used to build both the UC registered model name
# (`lensiq_<slug>`) and the serving endpoint name (`lensiq-<slug-with-dashes>`).
# Match the model ids declared in client/src/lib/models.ts so the AppKit
# server can bind the same alias.
dbutils.widgets.text("model_slug", "license_plate")
dbutils.widgets.text("model_display_name", "License plates")
# Roboflow Universe coordinates. The Universe URL for a project is
#   https://universe.roboflow.com/<workspace>/<project>
# and the model id you'd pass to `serverless.roboflow.com` is
# `<project>/<version>`. We need all three to download weights via the
# Roboflow SDK at deploy time: rf.workspace(WS).project(PROJ).version(VER).model.download()
dbutils.widgets.text("roboflow_workspace", "samrat-sahoo")
dbutils.widgets.text("roboflow_project", "license-plates-f8vsn")
dbutils.widgets.text("roboflow_version", "5")
# Roboflow API key for the weight download (deploy time only). At serve
# time the PyFunc reads weights from the MLflow artifact, never the API.
dbutils.widgets.text("api_key_scope", "reggie_pierce")
dbutils.widgets.text("api_key_secret", "ROBOFLOW_API_KEY")

# Optional output filters applied AFTER local YOLO inference. Public
# Universe models are noisy on out-of-distribution CCTV (false-positive
# bboxes spanning whole frames, single-class projects whose class label
# doesn't match our endpoint name, etc). These filters let us reuse a noisy
# Universe model without retraining by post-processing its predictions:
#
#   - class_label_override: rename every kept prediction's class to this
#     fixed string. Use it to align an upstream class like "sign" or
#     "Water-u6Vi" with the endpoint's slug ("wet_floor_sign", "spill"),
#     keeping the cross-endpoint contract uniform.
#   - min_confidence: drop predictions below this absolute confidence.
#     Acts as a floor on what YOLO returns: the PyFunc passes
#     `max(request_conf, min_confidence)` to `YOLO.predict(conf=...)` so
#     the model never bothers scoring anything below the bar.
#   - min_area_pct / max_area_pct: drop bboxes covering less than (more than)
#     N% of the frame. Reject edge-sliver micro-detections (low) and
#     hallucinated frame-spanning boxes (high).
#   - min_y_center_pct: drop bboxes whose vertical center is in the top N%
#     of the frame. Cones and spills are floor-level; this kills shelf-
#     signage / ceiling false positives without affecting real detections.
#
# All filters are optional; pass `""` (or 0 / 100 for the area bounds) to
# disable. Defaults below are no-ops so the existing detectors keep their
# current behaviour.
dbutils.widgets.text("class_label_override", "")
dbutils.widgets.text("min_confidence", "0.0")
dbutils.widgets.text("min_area_pct", "0.0")
dbutils.widgets.text("max_area_pct", "100.0")
dbutils.widgets.text("min_y_center_pct", "0.0")

In [ ]:
%pip install -q mlflow>=2.13 ultralytics==8.3.0 roboflow>=1.1.0 pillow "numpy<2"
dbutils.library.restartPython()

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
LOG = logging.getLogger("deploy_roboflow_detector")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
MODEL_SLUG = dbutils.widgets.get("model_slug").strip()
MODEL_DISPLAY_NAME = dbutils.widgets.get("model_display_name").strip() or MODEL_SLUG
ROBOFLOW_WORKSPACE = dbutils.widgets.get("roboflow_workspace").strip()
ROBOFLOW_PROJECT = dbutils.widgets.get("roboflow_project").strip()
ROBOFLOW_VERSION = int(dbutils.widgets.get("roboflow_version").strip())
API_KEY_SCOPE = dbutils.widgets.get("api_key_scope")
API_KEY_SECRET = dbutils.widgets.get("api_key_secret")

# Optional post-filter knobs. Empty string / no-op default values mean
# "filter disabled" and reproduce the previous notebook's behaviour
# verbatim for the detectors that don't need filtering (license_plate,
# cigarette_vape, slip_fall).
CLASS_LABEL_OVERRIDE = dbutils.widgets.get("class_label_override").strip()
MIN_CONFIDENCE = float(dbutils.widgets.get("min_confidence") or 0.0)
MIN_AREA_PCT = float(dbutils.widgets.get("min_area_pct") or 0.0)
MAX_AREA_PCT = float(dbutils.widgets.get("max_area_pct") or 100.0)
MIN_Y_CENTER_PCT = float(dbutils.widgets.get("min_y_center_pct") or 0.0)

# Conventions tying everything together:
#   - UC registered model:  <catalog>.<schema>.lensiq_<slug>
#   - Serving endpoint:     lensiq-<slug-with-dashes>
# The AppKit server expects matching aliases (see server/server.ts).
REGISTERED = f"{CATALOG}.{SCHEMA}.lensiq_{MODEL_SLUG}"
ENDPOINT = f"lensiq-{MODEL_SLUG.replace('_', '-')}"

# Sanity-check the Roboflow key exists before we waste time downloading
# weights. The key is only used at deploy time; the served PyFunc never
# touches Roboflow.
API_KEY = dbutils.secrets.get(scope=API_KEY_SCOPE, key=API_KEY_SECRET)
if not API_KEY:
    raise RuntimeError(
        f"Roboflow API key not found at secrets/{API_KEY_SCOPE}/{API_KEY_SECRET}"
    )

LOG.info("Deploying %s -> %s", MODEL_DISPLAY_NAME, ENDPOINT)
LOG.info("  universe: %s/%s/%s", ROBOFLOW_WORKSPACE, ROBOFLOW_PROJECT, ROBOFLOW_VERSION)
LOG.info("  registered_model=%s", REGISTERED)
LOG.info("  filters: relabel=%r min_conf=%.2f area=[%.2f-%.2f]%% min_y_center=%.0f%%",
         CLASS_LABEL_OVERRIDE or None, MIN_CONFIDENCE,
         MIN_AREA_PCT, MAX_AREA_PCT, MIN_Y_CENTER_PCT)

## Download trained weights from Roboflow Universe

Pulls the `weights.pt` for `{workspace}/{project}/{version}` using the
Roboflow Python SDK. This is the **only** Roboflow API call in the
deployment lifecycle - the served PyFunc never touches Roboflow.

The Roboflow SDK writes weights into a project-versioned subdirectory
under the current working directory; we run the download inside a fresh
temp dir and then glob for `**/*.pt` to find the file regardless of how
the SDK names it.

In [ ]:
import glob
import os
import pathlib
import shutil
import tempfile

from roboflow import Roboflow

_DOWNLOAD_DIR = pathlib.Path(tempfile.mkdtemp(prefix=f"rfw_{MODEL_SLUG}_"))
_PREV_CWD = os.getcwd()
os.chdir(_DOWNLOAD_DIR)
try:
    LOG.info("Downloading weights to %s", _DOWNLOAD_DIR)
    _rf = Roboflow(api_key=API_KEY)
    _project = _rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    _version = _project.version(ROBOFLOW_VERSION)
    _version.model.download()
finally:
    os.chdir(_PREV_CWD)

# The SDK names the weights file inconsistently across project types
# (most YOLO projects yield `weights.pt`, some land at `weights/best.pt`).
# Glob for any .pt under the download dir and take the largest, which is
# always the weights file (not the tiny optimizer state).
_pt_files = sorted(
    glob.glob(str(_DOWNLOAD_DIR / "**" / "*.pt"), recursive=True),
    key=lambda p: pathlib.Path(p).stat().st_size,
    reverse=True,
)
if not _pt_files:
    raise RuntimeError(
        f"No .pt weights file found under {_DOWNLOAD_DIR}. "
        f"This usually means the Roboflow project '{ROBOFLOW_WORKSPACE}/"
        f"{ROBOFLOW_PROJECT}/{ROBOFLOW_VERSION}' wasn't trained with a YOLO "
        f"architecture or the owner disabled weight downloads. Either pick a "
        f"different Universe model or download the dataset and train a YOLO "
        f"model on Databricks."
    )

# Copy to a stable, absolute path that won't get garbage-collected before
# log_model bakes it into the MLflow artifact. /local_disk0 is the standard
# writable scratch on Databricks clusters; /tmp works on serverless too.
_writable_root = pathlib.Path("/local_disk0") if pathlib.Path("/local_disk0").exists() else pathlib.Path("/tmp")
LOCAL_WEIGHTS = str(_writable_root / f"rfw_{MODEL_SLUG}_weights.pt")
shutil.copyfile(_pt_files[0], LOCAL_WEIGHTS)
LOG.info("Weights ready at %s (%.1f MB)",
         LOCAL_WEIGHTS, pathlib.Path(LOCAL_WEIGHTS).stat().st_size / 1024 / 1024)


## PyFunc wrapper

Wraps `ultralytics.YOLO(weights.pt)` for local inference inside the
serving container - identical pattern to `deploy_yolo.ipynb`'s
`YoloDetector` for the COCO model, with the addition of the post-filter
knobs (relabel, min confidence, area, vertical position) that the proxy
PyFunc used to apply to Roboflow's JSON response. No network calls are
made at predict time.

A few important MLflow + ultralytics gotchas this wrapper handles:

- **`YOLO_CONFIG_DIR`** is set in `load_context` *before* importing
  ultralytics. The default location (`~/.config/Ultralytics`) is
  read-only inside Databricks Model Serving containers; ultralytics
  writes a `settings.yaml` at import time and crashes if it can't.
- **Weights live in `context.artifacts["weights"]`** - an absolute path
  inside the serving container that MLflow materializes from the logged
  artifact at load time. Don't use the deploy-time path.
- **Post-filter floor** is applied by passing `max(request_conf, min_confidence)`
  to `YOLO.predict(conf=...)` so the model doesn't waste cycles scoring
  detections we're about to throw away.

In [ ]:
import base64
import io
import os
import tempfile
import uuid

import mlflow
import mlflow.pyfunc
import pandas as pd
from mlflow.models import infer_signature


def _writable_yolo_config_dir():
    """Return a writable directory for ultralytics settings.yaml.

    Ultralytics writes its settings.yaml at import time. The default
    `~/.config/Ultralytics` is read-only inside Databricks Model Serving
    containers, which makes `from ultralytics import YOLO` crash. We
    probe Spark local dirs first (writable, low-latency) and fall back
    to tempfile.gettempdir() (always writable on serving).
    """
    candidates = []
    spark_local_dirs = os.environ.get("SPARK_LOCAL_DIRS")
    if spark_local_dirs:
        candidates.extend(spark_local_dirs.split(","))
    candidates.append(tempfile.gettempdir())
    for base in candidates:
        try:
            target = os.path.join(base, f"yolo_config_{uuid.uuid4().hex}")
            os.makedirs(target, exist_ok=True)
            probe = os.path.join(target, ".probe")
            with open(probe, "w") as f:
                f.write("")
            os.remove(probe)
            return target
        except OSError:
            continue
    raise RuntimeError("No writable directory found for YOLO_CONFIG_DIR")


class RoboflowDetector(mlflow.pyfunc.PythonModel):
    """PyFunc that runs a Roboflow Universe YOLO model locally via
    ultralytics. Weights are baked into the MLflow artifact at deploy
    time; the served PyFunc makes zero outbound network calls.

    Inputs (per row):
      - image: base64-encoded JPEG/PNG (with or without `data:` prefix).
      - conf:  optional confidence threshold, default 0.35.
      - iou:   optional IoU threshold for NMS, default 0.5.

    Output (per row): list of `{label, class_id, confidence, bbox}` with
    `bbox` as `[x1, y1, x2, y2]`. Matches the YOLO endpoint's contract.

    Optional output filters (read from `model_config`):
      - class_label_override: rename every kept prediction's class to a
        fixed string. Use when an upstream Universe model's only class
        ("sign", "Water-u6Vi", ...) doesn't match this endpoint's slug.
      - min_confidence: floor on confidence. Passed to YOLO.predict()
        as the conf threshold (so the model never bothers scoring boxes
        below it) and rechecked in post-filter for safety.
      - min_area_pct / max_area_pct: drop bboxes whose area is outside
        this fraction-of-frame range. Kills edge slivers and hallucinated
        full-frame boxes that some out-of-distribution models emit.
      - min_y_center_pct: drop bboxes whose vertical center is in the top
        N% of the frame. Cones and spills are floor-level; this rejects
        shelf signage / ceiling false positives.

    These filters live INSIDE the served PyFunc (instead of in the AppKit
    server) so a single notebook + bundle entry can ship a noisy Universe
    model as a clean single-purpose detector. The downstream contract
    that the AppKit server consumes never changes.
    """

    def load_context(self, context):
        os.environ["YOLO_CONFIG_DIR"] = _writable_yolo_config_dir()
        from ultralytics import YOLO
        from PIL import Image

        self._YOLO = YOLO
        self._Image = Image
        self._model = YOLO(context.artifacts["weights"])
        self._names = self._model.names

        cfg = context.model_config or {}
        self._class_label_override = (cfg.get("class_label_override") or "").strip() or None
        self._min_confidence = float(cfg.get("min_confidence") or 0.0)
        self._min_area_pct = float(cfg.get("min_area_pct") or 0.0)
        self._max_area_pct = float(cfg.get("max_area_pct") or 100.0)
        self._min_y_center_pct = float(cfg.get("min_y_center_pct") or 0.0)

    def _decode(self, image_b64):
        if not image_b64:
            return None
        if isinstance(image_b64, str) and image_b64.startswith("data:"):
            image_b64 = image_b64.split(",", 1)[1]
        return self._Image.open(io.BytesIO(base64.b64decode(image_b64))).convert("RGB")

    def _keep(self, x1, y1, x2, y2, image_w, image_h):
        """Apply geometric post-filters; confidence floor is enforced upstream
        in YOLO.predict(conf=...). Returns True if the box survives."""
        if not image_w or not image_h:
            return True
        w = max(0, x2 - x1)
        h = max(0, y2 - y1)
        area_pct = (w * h) / (image_w * image_h) * 100.0
        if area_pct < self._min_area_pct or area_pct > self._max_area_pct:
            return False
        y_center_pct = ((y1 + y2) / 2.0) / image_h * 100.0
        if y_center_pct < self._min_y_center_pct:
            return False
        return True

    def _run_one(self, image_b64, conf, iou):
        img = self._decode(image_b64)
        if img is None:
            return []
        import numpy as np

        arr = np.array(img)
        image_h, image_w = arr.shape[:2]
        effective_conf = max(float(conf or 0.35), self._min_confidence)
        results = self._model.predict(
            source=arr,
            conf=max(0.01, effective_conf),
            iou=float(iou or 0.5),
            verbose=False,
        )
        out = []
        for r in results:
            if r.boxes is None:
                continue
            xyxy = r.boxes.xyxy.cpu().numpy()
            cls = r.boxes.cls.cpu().numpy()
            conf_arr = r.boxes.conf.cpu().numpy()
            for box, c, s in zip(xyxy, cls, conf_arr):
                x1, y1, x2, y2 = (int(round(v)) for v in box.tolist())
                if not self._keep(x1, y1, x2, y2, image_w, image_h):
                    continue
                c = int(c)
                label = self._class_label_override or self._names.get(c, str(c))
                out.append({
                    "label": label,
                    "class_id": c,
                    "confidence": float(s),
                    "bbox": [x1, y1, x2, y2],
                })
        return out

    def predict(self, context, model_input, params=None):
        if hasattr(model_input, "to_dict"):
            rows = model_input.to_dict(orient="records")
        elif isinstance(model_input, dict):
            rows = [model_input]
        else:
            rows = list(model_input)
        return [self._run_one(r.get("image"), r.get("conf"), r.get("iou")) for r in rows]

## Log + register

In [ ]:
_TINY_PNG_B64 = (
    "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8/5+hHgAH"
    "ggJ/PchI7wAAAABJRU5ErkJggg=="
)

sample_input = pd.DataFrame([
    {"image": _TINY_PNG_B64, "conf": 0.35, "iou": 0.5},
])
sample_output = [[]]
signature = infer_signature(sample_input, sample_output)

mlflow.set_registry_uri("databricks-uc")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

# model_config carries only the static post-filter knobs. The weights
# travel through `artifacts` (materialized into the serving container by
# MLflow on load) and the Roboflow API key is intentionally NOT included
# anywhere - it was only needed at download time above.
model_config = {
    "class_label_override": CLASS_LABEL_OVERRIDE,
    "min_confidence": MIN_CONFIDENCE,
    "min_area_pct": MIN_AREA_PCT,
    "max_area_pct": MAX_AREA_PCT,
    "min_y_center_pct": MIN_Y_CENTER_PCT,
}

with mlflow.start_run(run_name=f"deploy_roboflow_{MODEL_SLUG}") as run:
    info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=RoboflowDetector(),
        artifacts={"weights": LOCAL_WEIGHTS},
        signature=signature,
        input_example=sample_input,
        registered_model_name=REGISTERED,
        model_config=model_config,
        pip_requirements=[
            "mlflow>=2.13",
            "ultralytics==8.3.0",
            "torch>=2.0.0",
            "torchvision>=0.15.0",
            "numpy<2",
            "pillow",
        ],
    )
    mlflow.set_tag("lensiq.detector_slug", MODEL_SLUG)
    mlflow.set_tag("lensiq.display_name", MODEL_DISPLAY_NAME)
    mlflow.set_tag("lensiq.roboflow_workspace", ROBOFLOW_WORKSPACE)
    mlflow.set_tag("lensiq.roboflow_project", ROBOFLOW_PROJECT)
    mlflow.set_tag("lensiq.roboflow_version", str(ROBOFLOW_VERSION))
    if CLASS_LABEL_OVERRIDE:
        mlflow.set_tag("lensiq.class_label_override", CLASS_LABEL_OVERRIDE)
LOG.info("Logged model URI: %s", info.model_uri)

## Create / update the serving endpoint

The endpoint serves the locally-loaded YOLO model with no secrets or
external env vars - the weights are inside the MLflow artifact and the
PyFunc reads them via `context.artifacts["weights"]` at load time.

First-time endpoint creation builds a container image with `ultralytics`
+ `torch`, which takes ~5-10 minutes. Subsequent config updates redeploy
the existing image and only take ~1-2 minutes.

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

client = mlflow.MlflowClient()
versions = client.search_model_versions(f"name='{REGISTERED}'")
latest_version = max(versions, key=lambda v: int(v.version)).version
LOG.info("Deploying %s version %s -> endpoint %s", REGISTERED, latest_version, ENDPOINT)

served = ServedEntityInput(
    entity_name=REGISTERED,
    entity_version=latest_version,
    workload_size="Small",
    scale_to_zero_enabled=True,
)

w = WorkspaceClient()
try:
    w.serving_endpoints.get(name=ENDPOINT)
    LOG.info("Endpoint exists; updating config")
    w.serving_endpoints.update_config(name=ENDPOINT, served_entities=[served])
except Exception:
    LOG.info("Endpoint not found; creating")
    w.serving_endpoints.create(
        name=ENDPOINT,
        config=EndpointCoreConfigInput(name=ENDPOINT, served_entities=[served]),
    )
LOG.info("Submitted deployment for %s; watch Serving UI for readiness.", ENDPOINT)